# Burstchester CLI Colab Test Training

이 노트북은 `https://github.com/tomongoose/burstchester-cli`를 Colab에 클론한 뒤, Burstchester dataset ID로 모델 훈련을 테스트한다.

필요한 준비물:

- Burstchester 웹에서 발급한 CLI access token
- 훈련에 사용할 Burstchester dataset ID 1개 이상
- 선택 사항: gated Hugging Face 모델 또는 모델 업로드에 사용할 `HF_TOKEN`

Colab 런타임은 `Runtime > Change runtime type > GPU`로 설정하는 것을 권장한다.

In [ ]:
#@title 1. Training settings
REPO_URL = "https://github.com/tomongoose/burstchester-cli.git" #@param {type:"string"}
REPO_DIR = "/content/burstchester-cli" #@param {type:"string"}
DATASET_IDS = "" #@param {type:"string"}
BASE_MODEL = "Qwen/Qwen3-0.6B" #@param {type:"string"}
TRAIN_COMMAND = "train" #@param ["train", "train-gemma-2b-it-lora", "train-gemma4-e2b-full"]
TRAINING_METHOD = "qlora" #@param ["qlora", "lora", "full"]
WORKSPACE = "/content/burstchester-training" #@param {type:"string"}
EPOCHS = "1" #@param {type:"string"}
BATCH_SIZE = "1" #@param {type:"string"}
MAX_SEQ_LENGTH = "128" #@param {type:"string"}
LORA_RANK = "8" #@param {type:"string"}
LORA_ALPHA = "16" #@param {type:"string"}
LORA_DROPOUT = "0.05" #@param {type:"string"}
SKIP_REGISTER = True #@param {type:"boolean"}
OUTPUT_MODEL_REPO = "" #@param {type:"string"}
MODEL_POINT_COST = "100" #@param {type:"string"}

print("Settings loaded. Secrets are requested in the next cell.")

In [ ]:
# 2. Enter secrets without saving them in the notebook output.
import os
from getpass import getpass

if not DATASET_IDS.strip():
    raise ValueError("Set DATASET_IDS before continuing.")

access_token = os.environ.get("BURSTCHESTER_ACCESS_TOKEN") or getpass("Burstchester CLI access token: ")
if not access_token.strip():
    raise ValueError("Burstchester access token is required.")
os.environ["BURSTCHESTER_ACCESS_TOKEN"] = access_token.strip()

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    entered = getpass("Hugging Face token (optional, press Enter to skip): ")
    hf_token = entered.strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

print("Secrets configured in environment.")

In [ ]:
# 3. Clone or update the standalone CLI repository.
import os
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "status", "--short", "--branch"], cwd=REPO_DIR, check=True)

In [ ]:
# 4. Install Node 20 and Python training dependencies.
# Colab images may ship an older Node version, while the CLI requires Node >= 20.
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!python -m pip install -q -U torch transformers peft accelerate datasets bitsandbytes huggingface_hub

In [ ]:
# 5. Import dataset IDs into the CLI local dataset list.
from pathlib import Path
import re
import subprocess

repo = Path(REPO_DIR)
dataset_ids = list(dict.fromkeys([value for value in re.split(r"[\s,]+", DATASET_IDS.strip()) if value]))
dataset_file = Path(WORKSPACE).with_suffix(".dataset-ids.txt")
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_file.write_text("\n".join(dataset_ids) + "\n", encoding="utf-8")

subprocess.run(
    ["node", "src/cli.mjs", "dataset-list", "import", "--file", str(dataset_file)],
    cwd=repo,
    check=True,
)
subprocess.run(["node", "src/cli.mjs", "dataset-list", "show"], cwd=repo, check=True)

In [ ]:
# 6. Run a preflight check. This validates dataset download access before training starts.
import subprocess

preflight_cmd = [
    "node", "src/cli.mjs", TRAIN_COMMAND,
    "--model-repo", BASE_MODEL,
    "--workspace", WORKSPACE,
    "--python", "python",
    "--preflight-only",
]
if TRAIN_COMMAND == "train":
    preflight_cmd += ["--training-method", TRAINING_METHOD]

subprocess.run(preflight_cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 7. Train the model. For a quick test, keep epochs/batch/sequence length small.
import subprocess

train_cmd = [
    "node", "src/cli.mjs", TRAIN_COMMAND,
    "--model-repo", BASE_MODEL,
    "--workspace", WORKSPACE,
    "--python", "python",
    "--epochs", EPOCHS,
    "--batch-size", BATCH_SIZE,
    "--max-seq-length", MAX_SEQ_LENGTH,
    "--lora-rank", LORA_RANK,
    "--lora-alpha", LORA_ALPHA,
    "--lora-dropout", LORA_DROPOUT,
]
if TRAIN_COMMAND == "train":
    train_cmd += ["--training-method", TRAINING_METHOD]

subprocess.run(train_cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 8. Optional: upload the trained output to Hugging Face and register it in Burstchester.
# Leave SKIP_REGISTER=True for a training-only smoke test.
from pathlib import Path
import os
import subprocess

if SKIP_REGISTER:
    print("Skipping upload/register. Trained output should be under:", Path(WORKSPACE) / "output")
else:
    if not OUTPUT_MODEL_REPO.strip():
        raise ValueError("Set OUTPUT_MODEL_REPO or set SKIP_REGISTER=True.")
    if not os.environ.get("HF_TOKEN"):
        raise ValueError("HF_TOKEN is required to upload to Hugging Face.")

    from huggingface_hub import HfApi, create_repo
    output_dir = str(Path(WORKSPACE) / "output")
    create_repo(repo_id=OUTPUT_MODEL_REPO, token=os.environ["HF_TOKEN"], exist_ok=True)
    HfApi(token=os.environ["HF_TOKEN"]).upload_folder(
        folder_path=output_dir,
        repo_id=OUTPUT_MODEL_REPO,
        repo_type="model",
    )
    model_url = f"https://huggingface.co/{OUTPUT_MODEL_REPO}/resolve/main/adapter_model.safetensors"

    register_cmd = [
        "node", "src/cli.mjs", "register-model",
        "--huggingface-url", model_url,
        "--base-model", BASE_MODEL,
        "--dataset-file", str(dataset_file),
        "--training-method", TRAINING_METHOD if TRAIN_COMMAND == "train" else ("full" if TRAIN_COMMAND == "train-gemma4-e2b-full" else "lora"),
        "--point-cost", MODEL_POINT_COST,
    ]
    subprocess.run(register_cmd, cwd=REPO_DIR, check=True)